# Meqpy Tutorial — 5c. Molecule Examples

[← Previous: 5b. Molecule](05b_Molecule.ipynb) | [🏠 Index](00_Overview.ipynb) | *(end)*

In [ ]:
import meqpy

import numpy as np
import matplotlib.pyplot as plt

### Overview

- [5. Spatial Resolution](#spatial)
    - 5.1 Cube class [→ 5a. Cubes and Transitions](05a_Cubes_and_Transitions.ipynb)
        - 5.1.1 Cubes from file
        - 5.1.2 Cubes from 2p<sub>z</sub> vector
    - 5.2 Transition Class [→ 5a. Cubes and Transitions](05a_Cubes_and_Transitions.ipynb)
        - 5.2.1 Dyson
            - Coordinates and Slice Height
            - Properties and Methods
    - 5.3 Molecule [→ 5b Molecule](05b_Molecule.ipynb)
        - 5.3.1 Charging Transitions: Dyson
            - Add Dyson Transition
            - Missing Dyson Transitions
            - Molecule and Dyson Shape
            - Dyson Amplitudes
        - 5.3.2 Helper Functions
        - 5.3.3 Charging Rates
            - Coupling to plane wave Sample
            - Coupling to *s*-wave Tip
            - Point Spectroscopy
    - [5.4 Example Experiments](#spatial_example)
        - [5.4.1 Constant Height STS](#spatial_example_chSTS)
        - [5.4.2 I(V) Point Spectroscopy](#spatial_example_pointspec)
        - [5.4.3 Constant Height STML Mapping](#spatial_example_STML)
        - [5.4.4 Constant Height Photocurrent](#spatial_example_PC)

<a id='spatial'></a>
## 5. Spatial Resolution
Another extension of the ``System`` class is the ``Molecule`` class, which allows systems to be solved with spatial resolution. For this, molecular orbitals can be used to determine the spatial dependence of various transition rates — for example, Dyson orbitals for the charge transitions. These orbitals must be computed by external means, e.g. using DFT or tight-binding methods, and then loaded into ``meqpy`` via the ``Cube`` class. A ``Cube`` object can then be used to instantiate a ``Transition`` object, such as ``Dyson``, which handles the charging transition rates in a ``Molecule``.

In this chapter, we have so far discussed the ``Cube`` class, the ``Transition`` and ``Dyson`` classes and the ``Molecule`` class. In the following we will show some examples on how to combine them all and run some in silico experiments.

<a id='spatial_example'></a>
### 5.4 Example Experiments

In the following we will look at a few examples on how to use the ``Molecule`` class with ``Dyson`` transitions.


<a id='spatial_example_chSTS'></a>
#### 5.4.1 Constant Height STS
First we will consider the simple case of a naphthalene molecule and its frontier orbitals, namely HOMO and LUMO. 
We will take constant height maps and record the current as the measurement signal.

In [ ]:
# instantiate Molecule
molecule = meqpy.Molecule(
    hwhm=0.1,
    reorg_shift=0.25,
    kappa_mode="constant",
)

# add states
molecule.states = [
    meqpy.State("GS", energy=0.0, charge=0, multiplicity=1),
    meqpy.State("PIR", energy=1.5, charge=+1, multiplicity=2),
    meqpy.State("NIR", energy=1.0, charge=-1, multiplicity=2),
]

# add dysons
molecule.dyson_dict = {
    ("GS", "PIR"): meqpy.Dyson("./tutorial_files/naphthalene_homo.cub"),
    ("GS", "NIR"): meqpy.Dyson("./tutorial_files/naphthalene_lumo.cub"),
}

In [ ]:
# build rate matrix
tip_height = 7.0
molecule.tip_radius = 2.0

sample_distance = 1.0

bias = np.array([-2.0, +1.5])

# coupling to tip: with spatial information
Wt = molecule.charging_rates_dyson(tip_height, bias)

# coupling to sample: no spatial information
Ws = molecule.charging_rates(sample_distance, 0.0)

W = Wt + Ws
W.shape

In [ ]:
# solve rate equation to obtain probability vector P
molecule_P = meqpy.solve_equilibrium(W)

molecule_P.shape

In [ ]:
# measurement: current
current_operator = molecule.dQ * Ws * meqpy.constants.ELEMENTARY_CHARGE
molecule_current = meqpy.measurement(current_operator, molecule_P)
molecule_current.shape

In [ ]:
# display results
from mpl_toolkits.axes_grid1 import make_axes_locatable

fig, ax = plt.subplots(1, 2, figsize=(7, 3))

for i in range(2):
    mesh = ax[i].pcolormesh(
        molecule.y, molecule.x, molecule_current[..., i] * 1e12, cmap="grey"
    )
    ax[i].set_xticks([])
    ax[i].set_yticks([])
    ax[i].axes.set_aspect("equal")
    ax[i].set_title(
        f"$\\Delta$z = {tip_height} Å, Bias = {bias[i]:+.01f} V", fontsize=10
    )

    divider = make_axes_locatable(ax[i])
    cax = divider.append_axes("right", size="4%", pad=0.15)

    fig.colorbar(mesh, cax, label="Current (pA)")

plt.tight_layout(pad=2)
plt.show()

<a id='spatial_example_pointspec'></a>
#### 5.4.2 I(V) Point Spectroscopy
In the second example we will perform constant height I(V) spectroscopy at certain points. We can either use the ``Molecule.charging_rates_pointspec`` method, which is less memory intensive, or the ``Molecule.charging_rates_dyson`` method, which is faster. We will demonstrate for both methods here.

First we select three points in the xy-plane and display them in the constant height images simulated above.

In [ ]:
# choose points in cartesian coordinats and plot on the charging rates
points_xy = [  # (x, y) in Angstrom
    (2.5, 2.2),
    (3.4, 1.1),
    (4.3, 0.0),
]

# ---------------------------------------------------------
# display points in map
fig, ax = plt.subplots(1, 2, figsize=(5, 3))

for i in range(2):
    mesh = ax[i].pcolormesh(
        molecule.y, molecule.x, molecule_current[..., i], cmap="grey"
    )
    ax[i].set_xticks([])
    ax[i].set_yticks([])
    ax[i].axes.set_aspect("equal")
    ax[i].set_title(
        f"$\\Delta$z = {tip_height} Å, Bias = {bias[i]:+.01f} V", fontsize=10
    )
    for x, y in points_xy:
        ax[i].plot(y, x, "o")


plt.tight_layout(pad=2)
plt.show()

In [ ]:
# translate points from cartesian coordinates to indices
points_idx = molecule.get_xy_indices(points_xy)
points_idx

In [ ]:
# define experiment
tip_height = 7.0
molecule.tip_radius = 2.0

sample_distance = 1.0

bias = np.linspace(-2, 2, 201)

Option 1: Use the ``Molecule.charging_rates_dyson`` method to calculate everything at once and then select the points of interest

In [ ]:
# coupling to sample: no spatial information
Ws = molecule.charging_rates(sample_distance, 0.0)

# coupling to tip: with spatial information
Wt = molecule.charging_rates_dyson(tip_height, bias)

# grab points of interest
i, j = zip(*points_idx)
Wt_idx = Wt[i, j]

W = Wt_idx + Ws
W.shape

Option 2: Use the ``Molecule.charging_rates_pointspec`` method.

In [ ]:
# coupling to sample: no spatial information
Ws = molecule.charging_rates(sample_distance, 0.0)

# coupling to tip: with spatial information
Wt_idx = molecule.charging_rates_pointspec(points_idx, tip_height, bias)

W = Wt_idx + Ws
W.shape

In [ ]:
# solve rate equation to obtain probability vector P
molecule_P = meqpy.solve_equilibrium(W)

molecule_P.shape

In [ ]:
# measurement: current
current_operator = molecule.dQ * Ws * meqpy.constants.ELEMENTARY_CHARGE
molecule_current = meqpy.measurement(current_operator, molecule_P)

# differentiate current -> dI/dV
molecule_didv = np.gradient(molecule_current, bias, axis=-1)

molecule_current.shape

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(10, 2.5))

for i in range(3):
    ax2 = ax[i].twinx()

    ax[i].plot(bias, molecule_didv[i] * 1e9, "-", color="k")
    ax2.plot(bias, molecule_current[i] * 1e12, "-", color="r")

    # dI/dV axis (left)
    ax[i].set_ylabel("dI/dV (nS)")

    # Current axis (right)
    ax2.yaxis.label.set_color("red")
    ax2.tick_params(axis="y", colors="red")
    ax2.set_ylabel("Current (pA)")

    # x-axis
    ax[i].set_xlabel("Bias (V)")
    ax[i].set_title(
        f"$r_{i}$ = ({points_xy[i][0]}Å, {points_xy[i][1]}Å, {tip_height}Å)",
        color=f"C{i}",
    )

fig.tight_layout(pad=1)
plt.show()

<a id='spatial_example_STML'></a>
#### 5.4.3 Constant Height STML Mapping

This example will simulate a constant height STML map, taken at certain bias voltages. The charge transitions will be handled by ``Dyson`` objects and the radiative transition will be calculated using the ``OpticalTransition`` class.

First, we need to define our system including a neutral excited state "S1".

In [ ]:
# instantiate Molecule
molecule = meqpy.Molecule(
    hwhm=0.1,
    reorg_shift=0.25,
    kappa_mode="constant",
)

# add states
molecule.states = [
    meqpy.State("S0", energy=0.0, charge=0, multiplicity=1),
    meqpy.State("S1", energy=1.5, charge=0, multiplicity=1),
    meqpy.State("D0+", energy=2.0, charge=+1, multiplicity=2),
    meqpy.State("D0-", energy=1.0, charge=-1, multiplicity=2),
]

In a simple approximation, we will be using canonical orbitals for the Dyson objects. For the "S1" $\leftrightarrow$ "D0±" transition, the HOMO and LUMO cubes need to swapped compared to the "S0" $\leftrightarrow$ "D0±" transitions.

In [ ]:
# load cube files:
homo_cube = meqpy.Cube("./tutorial_files/naphthalene_homo.cub")
lumo_cube = meqpy.Cube("./tutorial_files/naphthalene_lumo.cub")

# add dysons
molecule.dyson_dict = {
    ("S0", "D0+"): meqpy.Dyson(homo_cube),
    ("S0", "D0-"): meqpy.Dyson(lumo_cube),
    # S1 -> D0± transitions are reversed
    ("S1", "D0+"): meqpy.Dyson(lumo_cube),
    ("S1", "D0-"): meqpy.Dyson(homo_cube),
}

In [ ]:
# build rate matrix
tip_height = 7.0
molecule.tip_radius = 2.0

sample_distance = 1.0
bias = -2.5

# coupling to tip: with spatial information
Wt = molecule.charging_rates_dyson(tip_height, bias)

# coupling to sample: no spatial information
Ws = molecule.charging_rates(sample_distance, 0.0)

W_charging = Wt + Ws
W_charging.shape  # (nx, ny, num_states, num_states)

To built the spatial map of the radiative decay we need to define the maximal emission strength ``v_rad``, the spatial emission strength ``emission_map`` and the matrix ``rad_matrix`` to assign the the emission to the correct pair of states, i.e. "S1" $\rightarrow$ "S0".

The ``emission_map`` is being calculated with the ``OpticalTransition.emission_strength()`` method and multiplied with ``v_rad``. The ``rad_matrix`` can be obtained via ``System.matrix_by_states``. To add everything together, we have to construct a matrix, that can be added to  ``W_charging``. In this case the shape is ``(nx, ny, num_states, num_states)``, since we use only one bias voltage, which got squeezed out.

We can achieve that, by broadcasting ``emission_map`` with as many trailing ``None`` as required.

**Note:** ``mirror_plane`` does not have to have the same value as ``sample_distance``.

In [ ]:
z_pointcharge = tip_height + molecule.tip_radius
mirror_plane = -2.0  # mirror_plane is 2 Angstrom below center of molecule

# radiative coupling of 1meV --> rate is given by coupling / planck
v_rad = 1e-3 / meqpy.constants.PLANCK_EV

# calculate emission map and multiply with v_rad
tdo = meqpy.OpticalTransition("./tutorial_files/naphthalene_tdo.cub")
emission_map = tdo.emission_strength(mirror_plane, z_pointcharge)
emission_map *= v_rad

# get matrix with correct pair of states
rad_matrix = molecule.matrix_by_states("S1", "S0")

# broadcast emission_map to correct shape
Wrad = emission_map[..., None, None] * rad_matrix
Wrad.shape  # (nx, ny, num_states, num_states)

In [ ]:
# add charging and radiative transition matrices
W = W_charging + Wrad

# solve rate equation to obtain probability vector P
molecule_P = meqpy.solve_equilibrium(W)

molecule_P.shape

In [ ]:
# measurement: current
current_operator = molecule.dQ * Ws * meqpy.constants.ELEMENTARY_CHARGE
molecule_current = meqpy.measurement(current_operator, molecule_P)

# measurement: emission
rad_operator = Wrad
molecule_emission = meqpy.measurement(rad_operator, molecule_P)

In literature, the emission maps are often normalized by the current. To capture the same effect in the simulation, the maps are normalized as well, including a constant background current [[1], [2]].


[1]: https://doi.org/10.1021/acsnano.4c07136
[2]: https://doi.org/10.1021/acsnano.6c01502

In [ ]:
bg_current = 3e-12  # 3pA background current

molecule_emission_normed = molecule_emission / (np.abs(molecule_current) + bg_current)

In [ ]:
# display results
from mpl_toolkits.axes_grid1 import make_axes_locatable

fig, ax = plt.subplots(1, 3, figsize=(9, 3))

for i in range(3):
    ax[i].set_xticks([])
    ax[i].set_yticks([])
    ax[i].axes.set_aspect("equal")

x, y = molecule.x, molecule.y
mesh = ax[0].pcolormesh(y, x, molecule_current * 1e12, cmap="grey")
divider = make_axes_locatable(ax[0])
cax = divider.append_axes("right", size="4%", pad=0.15)
fig.colorbar(mesh, cax, label="Current (pA)")
ax[0].set_title("Current")

mesh = ax[1].pcolormesh(y, x, molecule_emission * 1e-6, cmap="magma")
divider = make_axes_locatable(ax[1])
cax = divider.append_axes("right", size="4%", pad=0.15)
fig.colorbar(mesh, cax, label="Emission Rate (cts $\\mu s^{-1}$)")
ax[1].set_title("Emission")

mesh = ax[2].pcolormesh(y, x, molecule_emission_normed * 1e-15, cmap="magma")
divider = make_axes_locatable(ax[2])
cax = divider.append_axes("right", size="4%", pad=0.15)
fig.colorbar(mesh, cax, label="Norm. Emission (kcts/pC)")
ax[2].set_title("Norm. Emission")

plt.tight_layout(pad=2)
plt.show()

<a id='spatial_example_PC'></a>
#### 5.4.4 Constant Height Photocurrent

This example will simulate a the inverse of the previous experiment: Light induced photo current in the STM junction [[3]]. For this we will start in the same way as before, just adjust the energies of "S1" and "D0+". The Dyson orbitals will be assigned as before.

[3]: https://doi.org/10.1038/s41467-025-61296-x

In [ ]:
# instantiate Molecule
molecule = meqpy.Molecule(
    hwhm=0.1,
    reorg_shift=0.25,
    kappa_mode="constant",
)

# add states
molecule.states = [
    meqpy.State("S0", energy=0.0, charge=0, multiplicity=1),
    meqpy.State("S1", energy=2.0, charge=0, multiplicity=1),
    meqpy.State("D0+", energy=1.5, charge=+1, multiplicity=2),
    meqpy.State("D0-", energy=1.0, charge=-1, multiplicity=2),
]

In [ ]:
# load cube files:
homo_cube = meqpy.Cube("./tutorial_files/naphthalene_homo.cub")
lumo_cube = meqpy.Cube("./tutorial_files/naphthalene_lumo.cub")

# add dysons
molecule.dyson_dict = {
    ("S0", "D0+"): meqpy.Dyson(homo_cube),
    ("S0", "D0-"): meqpy.Dyson(lumo_cube),
    # S1 -> D0± transitions are reversed
    ("S1", "D0+"): meqpy.Dyson(lumo_cube),
    ("S1", "D0-"): meqpy.Dyson(homo_cube),
}

In [ ]:
# build rate matrix
tip_height = 7.0
molecule.tip_radius = 2.0

sample_distance = 4.0
bias = np.linspace(-2.0, 2.0, 5)

# coupling to tip: with spatial information
Wt = molecule.charging_rates_dyson(tip_height, bias)

# coupling to sample: no spatial information
Ws = molecule.charging_rates(sample_distance, 0.0)

W_charging = Wt + Ws
W_charging.shape  # (nx, ny, num_bias, num_states, num_states)

For the radiative transition, we will use a simple approximation and assume the same excitation and emission rate between the two neutral states. This can easily done by setting ``symmetric=True`` when calling ``matrix_by_states()``.

In addition, we have to consider the fact, that the bias dimension is not squeezed out anymore. Therefore, we have to add an additional ``None`` for the broadcasting, to obtain the correct shape of ``Wrad``.

In [ ]:
z_pointcharge = tip_height + molecule.tip_radius
mirror_plane = -2.0  # mirror_plane is 2 Angstrom below center of molecule

# radiative coupling of 1meV --> rate is given by coupling / planck
v_rad = 1e-3 / meqpy.constants.PLANCK_EV

# calculate emission map and multiply with v_rad
tdo = meqpy.OpticalTransition("./tutorial_files/naphthalene_tdo.cub")
emission_map = tdo.emission_strength(mirror_plane, z_pointcharge)
emission_map *= v_rad

# get matrix with correct pair of states, this time symmetric
rad_matrix = molecule.matrix_by_states("S1", "S0", symmetric=True)

# broadcast emission_map: one more None for bias dimension
Wrad = emission_map[..., None, None, None] * rad_matrix
Wrad.shape  # (nx, ny, 1, num_states, num_states)

In [ ]:
# add charging and radiative transition matrices
W = W_charging + Wrad

# solve rate equation to obtain probability vector P
molecule_P = meqpy.solve_equilibrium(W)

molecule_P.shape

In [ ]:
# measurement: current
current_operator = molecule.dQ * Ws * meqpy.constants.ELEMENTARY_CHARGE
molecule_current = meqpy.measurement(current_operator, molecule_P)

In [ ]:
# display results
from mpl_toolkits.axes_grid1 import make_axes_locatable

maps = np.moveaxis(molecule_current, -1, 0)
num_panels = len(maps)

fig, ax = plt.subplots(1, num_panels, figsize=(12, 5))

x, y = molecule.x, molecule.y
for i, imap in enumerate(maps):
    imap = imap * 1e12
    vrange = np.max(np.abs(imap))
    mesh = ax[i].pcolormesh(y, x, imap, cmap="bwr", vmin=-vrange, vmax=vrange)
    divider = make_axes_locatable(ax[i])
    cax = divider.append_axes("right", size="4%", pad=0.15)
    fig.colorbar(mesh, cax, label="Current (pA)")
    ax[i].set_title(f"Bias: {bias[i]}V")

    ax[i].set_xticks([])
    ax[i].set_yticks([])
    ax[i].axes.set_aspect("equal")

plt.tight_layout(pad=2)
plt.show()

---

[← Previous: 5b. Molecule](05b_Molecule.ipynb) | [🏠 Index](00_Overview.ipynb) | *(end)*